In [12]:
import json
import numpy as np
from src.utils import parse_raw, simple_line_plot, fourier_transform_plot, segment_data, segment_plot
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import os
from glob import glob

In [13]:
raw_data_path = "raw_data/Leo/S4/"

recording_file_path = glob(os.path.join(raw_data_path, "*.txt"))[0]
metadata_file_path = glob(os.path.join(raw_data_path, "*.json"))[0]


In [14]:
with open(recording_file_path, 'r') as f:
    file_content = f.readlines()
    
header_json, df = parse_raw(file_content)

if os.path.exists(metadata_file_path):
    with open(metadata_file_path, 'r') as meta_file:
        metadata = json.load(meta_file)

df.head()

,timestamp,THORAX,X,Y
0,2025-09-19 10:34:44.300,13.006592,-0.600625,0.632868
1,2025-09-19 10:34:44.305,13.006592,-0.598750,0.628527
2,2025-09-19 10:34:44.310,12.994385,-0.604219,0.633643
3,2025-09-19 10:34:44.315,12.963867,-0.600625,0.630388
4,2025-09-19 10:34:44.320,12.939453,-0.603125,0.632248


In [15]:
fig = simple_line_plot(df)
fig

In [16]:
fig = fourier_transform_plot(df)
fig

In [17]:
autocorr_thorax = np.correlate(df['THORAX'], df['THORAX'], mode='full')
lags = np.arange(-len(df['THORAX']) + 1, len(df['THORAX']))

fig = px.line(x=lags, y=autocorr_thorax, title='Autocorrelation of THORAX')
fig.update_xaxes(title_text='Lag')
fig.update_yaxes(title_text='Autocorrelation')
fig

In [18]:
autocorr_X = np.correlate(df['X'], df['X'], mode='full')
lags = np.arange(-len(df['X']) + 1, len(df['X']))

fig = px.line(x=lags, y=autocorr_X, title='Autocorrelation of X')
fig.update_xaxes(title_text='Lag')
fig.update_yaxes(title_text='Autocorrelation')
fig

In [19]:
def energie_moyenne(df, column : str ="X"):
    # Find the closest dip to 0 in autocorr_X (excluding the central peak)
    mid = len(autocorr_X) // 2
    # Search for local minima around the center
    search_range = autocorr_X[mid-500:mid+500]
    dip_idx = np.argmin(search_range)
    closest_dip = dip_idx - 500  # relative to center
    window_size = 10 * abs(closest_dip)
    energies = df[column].rolling(window=window_size, min_periods=1).apply(lambda x: np.mean(x**2), raw=True)
    return energies

fig = px.line(energie_moyenne(df,"X"))
fig

In [20]:
print(json.dumps(metadata, indent=4))

{
    "studentId": "68411A",
    "sequenceDescription": "Descente lente de 2 \u00e9tages puis descente rapide de 2 \u00e9tages, puis repos",
    "sessionId": "S4",
    "sequences": [
        {
            "sequenceId": 1,
            "begin": 268,
            "end": 5732,
            "sequenceContext": "DESCENTE"
        },
        {
            "sequenceId": 2,
            "begin": 5750,
            "end": 11316,
            "sequenceContext": "REPOS"
        },
        {
            "sequenceId": 3,
            "begin": 11317,
            "end": 15241,
            "sequenceContext": "DESCENTE"
        },
        {
            "sequenceId": 4,
            "begin": 15242,
            "end": 21569,
            "sequenceContext": "REPOS"
        }
    ]
}


In [21]:
fig = segment_plot(df, metadata["sequences"])
fig

In [22]:
str(df["timestamp"].iloc[0])

'2025-09-19 10:34:44.300000'